In [ ]:
import cv2
import numpy as np
import tensorflow as tf

In [ ]:
# -----------------------------
# 1. Load TFLite Model
# -----------------------------
interpreter = tf.lite.Interpreter(model_path="D:\Python 11..coding\BRAIN.tflite")  # or best_int8.tflite
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_height = input_details[0]['shape'][1]
input_width = input_details[0]['shape'][2]

print("✅ Model Loaded")
print("Input shape:", input_details[0]['shape'])



In [ ]:
# Example class names (replace with your 8 tumor classes)
class_names = [
    "No Tumor", "Meningioma", "Glioma", "Pituitary", 
    "Other1", "Other2", "Other3", "Other4"
]

def test_single_image_full(image_path):
    print("\n🔍 Testing Single Image...\n")
    image = cv2.imread(image_path)
    original = image.copy()

    # Preprocess
    image_resized = cv2.resize(image, (input_width, input_height))
    image_resized = cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)
    image_resized = image_resized.astype(np.float32) / 255.0
    image_resized = np.expand_dims(image_resized, axis=0)

    # Run inference
    interpreter.set_tensor(input_details[0]['index'], image_resized)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    print("Raw output shape:", output.shape)

    h, w, _ = original.shape
    boxes = []

    # Loop through predictions
    for i in range(output.shape[2]):
        # Each prediction has [x, y, w, h, conf, class1, class2, ...]
        conf = output[0, 4, i]
        class_scores = output[0, 5:, i]  # Assuming last 8 are class scores
        class_id = np.argmax(class_scores)
        class_conf = class_scores[class_id]

        # Final confidence = objectness * class score
        final_conf = conf * class_conf

        if final_conf > 0.4:  # Threshold
            x, y, bw, bh = output[0, 0:4, i]
            x1 = int((x - bw / 2) * w)
            y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w)
            y2 = int((y + bh / 2) * h)
            boxes.append((x1, y1, x2, y2, class_id, final_conf))

    # Draw boxes
    for (x1, y1, x2, y2, class_id, conf) in boxes:
        cv2.rectangle(original, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(
            original,
            f"{class_names[class_id]} {conf:.2f}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

    cv2.imshow("TFLite YOLO Result", original)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"✅ Detections: {len(boxes)}")


In [ ]:
test_single_image_full("Y1.jpg")